In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange

sys.path.append("../../")
import biked_commons
from biked_commons.design_evaluation.design_evaluation import *
from biked_commons.design_evaluation import scoring
from biked_commons.resource_utils import split_datasets_path
from biked_commons.conditioning import conditioning

c:\Users\Lyle\mambaforge\envs\pytorch_clip\lib\site-packages\sklearn\base.py:329: UserWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.1.3. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [2]:
data = pd.read_csv(split_datasets_path("CLIP_X_test.csv"), index_col=0)

#sample 100
data = data.sample(100, random_state=0)

In [3]:
evaluator, requirement_names, requirement_types = construct_tensor_evaluator(StandardEvaluations, data.columns)
isobjective = torch.tensor(requirement_types) == 1


In [4]:
num_data = data.shape[0]
rider_condition = conditioning.sample_riders(num_data, split="test")
use_case_condition = conditioning.sample_use_case(num_data, split="test")
text_condition = conditioning.sample_text(num_data, split="test")

condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Text": text_condition}

In [5]:
scores = evaluator(torch.tensor(data.values, dtype=torch.float32), condition)

c:\Users\Lyle\Documents\Files\DeCoDE\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)


In [6]:
objective_scores = scores[:, isobjective]
constraint_scores = scores[:, ~isobjective]
objective_scores = scores[:, isobjective].detach().numpy()
constraint_scores = scores[:, ~isobjective].detach().numpy()

In [7]:
validity_mask = np.all(constraint_scores <= 0, axis=1)

In [8]:
np.max(np.sum(constraint_scores <= 0, axis=1))

12

In [9]:
print(scoring.hypervolume(objective_scores, constraint_scores))
print(scoring.constraint_satisfaction_rate(objective_scores, constraint_scores))

c:\Users\Lyle\Documents\Files\DeCoDE\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)


0.0
0.8307142857142857
